# 4. Attention analysis
Deletion tests (is attention faithful?) and attention statistics (pitch, duration, measure position, rests).

Run `1_train.ipynb` and `2_ablations_and_controls.ipynb` first: this notebook uses the trained models `full` and `no_meas`.

In [1]:
import torch

from constants import *
from utils import get_device
from faithfulness import run_deletion_tests, run_pitch_matched_deletion_tests
from attention_inspector import AttentionInspector

In [2]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True uses the models of the dry run in results_quick_test/
# and one deletion fraction only.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    deletion_fractions = [0.10]
    num_random_draws = 1
else:
    results_dir = RESULTS_DIR
    deletion_fractions = DELETION_FRACTIONS
    num_random_draws = NUM_RANDOM_DRAWS

cuda


## Deletion tests
A fraction of the notes of every test piece is deleted and the piece is classified again. Four deletion orders: most attended notes first, least attended first, highest saliency first, random.

In [3]:
curves, deletion_summary = run_deletion_tests(
    tag="full",
    device=device,
    seeds=SEEDS,
    fractions=deletion_fractions,
    num_random_draws=num_random_draws,
    results_dir=results_dir
)

[full s0] Base Acc: 0.9486 | Spearman(attention, saliency): mean 0.412, median 0.433
    delete  5% attn_top     Acc: 0.8800 | Flip rate: 0.126
    delete  5% attn_bottom  Acc: 0.9486 | Flip rate: 0.000
    delete  5% grad_top     Acc: 0.8514 | Flip rate: 0.143
    delete  5% random       Acc: 0.9474 | Flip rate: 0.011
    delete 10% attn_top     Acc: 0.7714 | Flip rate: 0.223
    delete 10% attn_bottom  Acc: 0.9486 | Flip rate: 0.000
    delete 10% grad_top     Acc: 0.8400 | Flip rate: 0.154
    delete 10% random       Acc: 0.9486 | Flip rate: 0.019
    delete 20% attn_top     Acc: 0.6514 | Flip rate: 0.343
    delete 20% attn_bottom  Acc: 0.9429 | Flip rate: 0.017
    delete 20% grad_top     Acc: 0.8057 | Flip rate: 0.189
    delete 20% random       Acc: 0.9371 | Flip rate: 0.031
    delete 40% attn_top     Acc: 0.4171 | Flip rate: 0.566
    delete 40% attn_bottom  Acc: 0.9371 | Flip rate: 0.040
    delete 40% grad_top     Acc: 0.7257 | Flip rate: 0.274
    delete 40% random       Ac

In [4]:
deletion_summary

,order,frac,acc_mean,acc_sd,flip_mean,flip_sd,n
0,attn_bottom,0.05,0.922286,0.022714,0.004571,0.004781,5
1,attn_bottom,0.10,0.917714,0.019209,0.010286,0.009390,5
2,attn_bottom,0.20,0.916571,0.016463,0.016000,0.007451,5
3,attn_bottom,0.40,0.897143,0.027994,0.058286,0.014228,5
4,attn_top,0.05,0.881143,0.026311,0.096000,0.032677,5
5,attn_top,0.10,0.804571,0.049553,0.185143,0.036723,5
6,attn_top,0.20,0.628571,0.060474,0.377143,0.054210,5
7,attn_top,0.40,0.352000,0.050077,0.649143,0.056366,5
8,grad_top,0.05,0.854857,0.014343,0.107429,0.025683,5
9,grad_top,0.10,0.820571,0.031194,0.150857,0.021609,5


Control: the most attended notes are mostly notes of a few pitches, so deleting them also removes these pitches. Here the same number of notes of the same pitches is deleted at random positions (`pitch_matched`) and compared with the deletion of the most attended notes (`attn_top`).

In [ ]:
pitch_matched_curves, pitch_matched_summary = run_pitch_matched_deletion_tests(
    tag="full",
    device=device,
    seeds=SEEDS,
    fractions=deletion_fractions,
    num_random_draws=num_random_draws,
    results_dir=results_dir
)

In [ ]:
pitch_matched_summary

## Attention statistics of the proposed model
First the note table is collected: one row per note of every test piece, with its attention weight.

In [5]:
ai = AttentionInspector(device, results_dir=results_dir)

In [6]:
note_table = ai.collect_note_table("full", seeds=SEEDS)
note_table.head()

[full s0] collected 74257 notes from 175 pieces
[full s1] collected 71061 notes from 175 pieces
[full s2] collected 72725 notes from 175 pieces
[full s3] collected 74888 notes from 175 pieces
[full s4] collected 78605 notes from 175 pieces


,f_name,makam,correct,T,t,alpha,norm_attn,high,pitch,pitch_aeu,is_rest,komma_class,dur,meas_pos,dur_bin,seed
0,hicaz--sarki--curcuna--gonul_yarasindan--selah...,hicaz,True,778,0,0.000115,0.089642,False,C#4,C#4,False,4,0.050000,start,Very short (≤1/16),0
1,hicaz--sarki--curcuna--gonul_yarasindan--selah...,hicaz,True,778,1,0.001873,1.457164,False,A#5,Bb4,False,45,0.050000,middle,Very short (≤1/16),0
2,hicaz--sarki--curcuna--gonul_yarasindan--selah...,hicaz,True,778,2,0.001420,1.104632,False,A,A,False,40,0.050000,middle,Very short (≤1/16),0
3,hicaz--sarki--curcuna--gonul_yarasindan--selah...,hicaz,True,778,3,0.004699,3.656137,True,A#5,Bb4,False,45,0.050000,middle,Very short (≤1/16),0
4,hicaz--sarki--curcuna--gonul_yarasindan--selah...,hicaz,True,778,4,0.001942,1.510818,False,A,A,False,40,0.666667,middle,Long (3/8–3/4],0


Pitch enrichment, duration, measure position and rests. The report is written to `attention/full_report.json`. It also holds the checks that are not printed here (the duration test over the four shorter bins, the tests with one value per distinct piece, the permutation check for the high-attention notes, measure position without the rests); notebook 5 shows them.

In [7]:
report, enrichment = ai.build_report("full", note_table)
ai.print_report(report, enrichment)

=== Duration bins (unit level, rests excluded) ===
units by bin: {'Very short (≤1/16)': 721, 'Short (1/16–3/16]': 854, 'Medium (3/16–3/8]': 811, 'Long (3/8–3/4]': 534, 'Very long (>3/4)': 214}
units with all 5 bins: 128 | Friedman chi2: 4.11 | p: 0.39 | Kendall W: 0.008
median of the per-unit median T*alpha, by bin:
    Very short (≤1/16): 0.270
     Short (1/16–3/16]: 0.301
     Medium (3/16–3/8]: 0.330
        Long (3/8–3/4]: 0.330
      Very long (>3/4): 0.305
pairs of bins:
    Very short (≤1/16) vs Short (1/16–3/16]    n=703 delta=-0.155 p_holm=8.79e-05
    Very short (≤1/16) vs Medium (3/16–3/8]    n=660 delta=-0.227 p_holm=5.61e-12
    Very short (≤1/16) vs Long (3/8–3/4]       n=400 delta=-0.155 p_holm=8.41e-06
    Very short (≤1/16) vs Very long (>3/4)     n=139 delta=-0.094 p_holm=0.374
     Short (1/16–3/16] vs Medium (3/16–3/8]    n=803 delta=-0.126 p_holm=4e-08
     Short (1/16–3/16] vs Long (3/8–3/4]       n=531 delta=-0.036 p_holm=0.000204
     Short (1/16–3/16] vs Very 

Share of notes, share of attention and enrichment of the most frequent pitches of one makam.

In [8]:
makam = "hicaz"
enrichment[enrichment["makam"] == makam].head(10)

,makam,pitch,pitch_aeu,komma_class,n_occ,occ_share,attn_share,enrichment,mean_norm_attn,attn_entropy_bits,n_pitches
0,hicaz,A,A,40,8659,0.183972,0.213634,1.161232,1.166549,2.036802,17
8,hicaz,D,D,9,7956,0.169036,0.016958,0.100319,0.101678,2.036802,17
11,hicaz,E,E,18,7091,0.150658,0.040035,0.265736,0.274867,2.036802,17
7,hicaz,C#4,C#4,4,5958,0.126586,0.076162,0.601661,0.580871,2.036802,17
2,hicaz,A#5,Bb4,45,5224,0.110991,0.564505,5.086054,5.224059,2.036802,17
15,hicaz,G,G,31,5094,0.108229,0.035188,0.325124,0.337664,2.036802,17
12,hicaz,F,F,22,2929,0.062230,0.005762,0.092598,0.097141,2.036802,17
14,hicaz,F#4,F#4,26,2753,0.058491,0.020814,0.355846,0.425335,2.036802,17
4,hicaz,B,B,49,452,0.009603,0.002985,0.310879,0.339933,2.036802,17
16,hicaz,G#4,G#4,35,280,0.005949,0.009199,1.546336,1.223958,2.036802,17


## The same statistics for the model without the measure-position input
If attention still differs between the start, the middle and the end of the measure, the difference does not come from the measure-position input.

In [9]:
note_table_no_meas = ai.collect_note_table("no_meas", seeds=SEEDS)
report_no_meas, enrichment_no_meas = ai.build_report("no_meas", note_table_no_meas)
ai.print_report(report_no_meas, enrichment_no_meas)

[no_meas s0] collected 74257 notes from 175 pieces
[no_meas s1] collected 71061 notes from 175 pieces
[no_meas s2] collected 72725 notes from 175 pieces
[no_meas s3] collected 74888 notes from 175 pieces
[no_meas s4] collected 78605 notes from 175 pieces
=== Duration bins (unit level, rests excluded) ===
units by bin: {'Very short (≤1/16)': 721, 'Short (1/16–3/16]': 854, 'Medium (3/16–3/8]': 811, 'Long (3/8–3/4]': 534, 'Very long (>3/4)': 214}
units with all 5 bins: 128 | Friedman chi2: 16.65 | p: 0.0023 | Kendall W: 0.033
median of the per-unit median T*alpha, by bin:
    Very short (≤1/16): 0.333
     Short (1/16–3/16]: 0.359
     Medium (3/16–3/8]: 0.412
        Long (3/8–3/4]: 0.429
      Very long (>3/4): 0.433
pairs of bins:
    Very short (≤1/16) vs Short (1/16–3/16]    n=703 delta=-0.158 p_holm=1.06e-06
    Very short (≤1/16) vs Medium (3/16–3/8]    n=660 delta=-0.303 p_holm=2.15e-19
    Very short (≤1/16) vs Long (3/8–3/4]       n=400 delta=-0.290 p_holm=4.79e-15
    Very shor